# Model Internals Dashboard

Comprehensive summary tools for quick model audits:
per-layer statistics, head classification, MLP utilization,
residual stream health, and bottleneck detection.

## Why This Matters

Model internals dashboard provides tools for systematic analysis and comparison of transformer internals. These diagnostics help identify unexpected behaviors, compare architectures, and build a comprehensive picture of how the model processes information.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.model_internals_dashboard import (
    layer_statistics,
    head_classification_summary,
    mlp_utilization,
    residual_stream_health,
    bottleneck_detection,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])

def metric_fn(logits, tokens):
    return jnp.mean(logits[-1])

print("Model ready")

## Layer Statistics

In [ ]:
result = layer_statistics(model, tokens)
for stats in result['per_layer']:
    l = stats['layer']
    norm = stats.get('resid_norm_mean', 0)
    entropy = stats.get('attn_entropy_mean', 0)
    sparsity = stats.get('mlp_sparsity', 0)
    print(f"Layer {l}: resid_norm={norm:.3f}, attn_entropy={entropy:.3f}, mlp_sparsity={sparsity:.1%}")
print(f"\nResidual growth: {result['summary']['resid_growth']:.2f}x")

## Head Classification

In [ ]:
result = head_classification_summary(model, tokens)
print(f"Category distribution: {result['category_counts']}")
print(f"\nTop heads per category:")
for cat, heads in result['top_heads_per_category'].items():
    print(f"  {cat}:")
    for head_id, score in heads[:3]:
        print(f"    {head_id}: {score:.3f}")

## MLP Utilization

In [ ]:
result = mlp_utilization(model, tokens)
print(f"Overall dead fraction: {result['overall_dead_fraction']:.1%}")
print(f"Overall sparsity: {result['overall_sparsity']:.1%}")
for stats in result['per_layer']:
    print(f"  Layer {stats['layer']}: dead={stats['dead_fraction']:.1%}, "
          f"mean_act={stats['mean_activation']:.4f}, sparsity={stats['sparsity']:.1%}")

## Residual Stream Health

In [ ]:
result = residual_stream_health(model, tokens)
print("Norm trajectory:")
for l, norm in enumerate(result['norm_trajectory']):
    bar = '#' * int(float(norm) * 5)
    print(f"  Layer {l}: {float(norm):.3f} {bar}")
print(f"\nComponent balance: {result['component_balance']}")
if result['health_warnings']:
    print(f"\nWarnings:")
    for w in result['health_warnings']:
        print(f"  ⚠ {w}")
else:
    print("\nNo health warnings.")

## Bottleneck Detection

In [ ]:
result = bottleneck_detection(model, tokens, metric_fn=metric_fn)
print(f"Rank profile:")
for l, rank in enumerate(result['rank_profile']):
    print(f"  Layer {l}: effective_rank={float(rank):.2f}")
print(f"\nBottleneck layers: {result['bottleneck_layers']}")
print(f"Min rank at layer: {result['min_rank_layer']}")
if len(result['ablation_profile']) > 0:
    print(f"\nAblation impact:")
    for l, impact in enumerate(result['ablation_profile']):
        print(f"  Layer {l}: {float(impact):.6f}")